# Logistic GD + scikit-learn – Solution

**Short name (GitHub):** `LogReg_GD_Sklearn`  
**Lab source:** C1_W3 Lab06 + Lab07  
Work the **Practice Skeleton** first. This notebook is the worked key: loop + vectorized alternates, extra practice, simulation, and audience write-ups.


## Inline cheat-sheet

| Item | Result / code |
|------|----------------|
| Lab06 gradient check | `dj_db≈0.498618`, `dj_dw≈[0.498333, 0.498839]` |
| GD 10k × α=0.1 from 0 | $w\approx(5.281, 5.078)$, $b\approx-14.222$, $J\approx0.0171$ |
| 1-D tumor | $w\approx4.75$, $b\approx-11.68$, 0.5-threshold $\approx2.46$ |
| sklearn `C=np.inf` | larger $|w|$ than 10k GD (optimizer + different stopping) but same acc = 1.0 |
| sklearn default L2 | much smaller $|w|$, still acc = 1.0 on 6 points |


In [ ]:
import copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Data


In [ ]:
df2 = pd.read_csv("data/logreg_gd_2d.csv")
X_train = df2[["x0", "x1"]].to_numpy(dtype=float)
y_train = df2["y"].to_numpy(dtype=float)
print("X_train shape:", X_train.shape)
print("y_train:", y_train)

fig, ax = plt.subplots(figsize=(5.0, 4.2))
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="#1f77b4", s=90, label="y = 0")
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="#d62728", marker="x", s=110, label="y = 1")
ax.set_xlim(0, 4); ax.set_ylim(0, 3.5)
ax.set_xlabel("$x_0$"); ax.set_ylabel("$x_1$")
ax.set_title("Lab 2-feature set (m = 6)")
ax.legend(frameon=False); ax.grid(True, alpha=0.3)
plt.show()


## 2. Sigmoid, cost, gradient, GD


In [ ]:
def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))


def compute_cost_logistic(X, y, w, b):
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f = np.clip(sigmoid(np.dot(X[i], w) + b), 1e-15, 1 - 1e-15)
        cost += -(y[i] * np.log(f) + (1 - y[i]) * np.log(1 - f))
    return cost / m


def compute_cost_logistic_vec(X, y, w, b):
    f = np.clip(sigmoid(X @ w + b), 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(f) + (1 - y) * np.log(1 - f)))


def compute_gradient_logistic(X, y, w, b):
    m, n = X.shape
    dj_dw = np.zeros(n)
    dj_db = 0.0
    for i in range(m):
        err = sigmoid(np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] += err * X[i, j]
        dj_db += err
    return dj_db / m, dj_dw / m


def compute_gradient_logistic_vec(X, y, w, b):
    err = sigmoid(X @ w + b) - y
    m = X.shape[0]
    return float(np.mean(err)), (X.T @ err) / m


print(sigmoid(np.array([-100.0, 0.0, 100.0])))


In [ ]:
w_tmp = np.array([2., 3.])
b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_train, y_train, w_tmp, b_tmp)
print("dj_db:", dj_db_tmp)
print("dj_dw:", dj_dw_tmp.tolist())

dj_db_v, dj_dw_v = compute_gradient_logistic_vec(X_train, y_train, w_tmp, b_tmp)
print("vec dj_db:", dj_db_v)
print("vec dj_dw:", dj_dw_v.tolist())
print("match?", np.allclose(dj_dw_v, dj_dw_tmp) and np.isclose(dj_db_v, dj_db_tmp))


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters, grad_fn=compute_gradient_logistic, verbose=True):
    J_history = []
    w = copy.deepcopy(np.asarray(w_in, dtype=float))
    b = float(b_in)
    for i in range(num_iters):
        dj_db, dj_dw = grad_fn(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        J_history.append(compute_cost_logistic(X, y, w, b))
        if verbose and i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}")
    return w, b, J_history


w_out, b_out, J_history = gradient_descent(
    X_train, y_train, np.zeros(2), 0.0, 0.1, 10000
)
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")
print("final J:", J_history[-1])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4))
axes[0].plot(J_history[:800], color="#1f77b4")
axes[0].set_title("J, first 800 steps"); axes[0].set_xlabel("iter"); axes[0].set_ylabel("J")
axes[0].grid(True, alpha=0.3)
axes[1].plot(np.log10(np.clip(J_history, 1e-12, None)), color="#d62728")
axes[1].set_title("log10 J, all 10k steps"); axes[1].set_xlabel("iter")
axes[1].grid(True, alpha=0.3)
fig.tight_layout(); plt.show()


In [ ]:
xx, yy = np.meshgrid(np.linspace(0, 4, 200), np.linspace(0, 3.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
prob = sigmoid(grid @ w_out + b_out).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
cs = ax.contourf(xx, yy, prob, levels=20, cmap="RdBu_r", vmin=0, vmax=1, alpha=0.85)
fig.colorbar(cs, ax=ax, fraction=0.046, label=r"$P(y=1)$")
ax.contour(xx, yy, prob, levels=[0.5], colors="k", linewidths=1.4)
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="#1f77b4", s=90, edgecolor="white", zorder=3, label="y=0")
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="#d62728", marker="x", s=110, zorder=3, label="y=1")
ax.set_xlim(0, 4); ax.set_ylim(0, 3.5)
ax.set_xlabel("$x_0$"); ax.set_ylabel("$x_1$")
ax.set_title("From-scratch GD boundary")
ax.legend(frameon=False); plt.show()

pred = (sigmoid(X_train @ w_out + b_out) >= 0.5).astype(float)
print("train acc GD:", np.mean(pred == y_train))


## 3. scikit-learn (Lab07)

Default `LogisticRegression` applies **L2**. On 6 linearly separable points both the regularized and unregularized models reach accuracy 1.0, but the coefficients differ a lot. Use `C=np.inf` for an unregularized comparison (sklearn ≥1.8 deprecates `penalty=None`).


In [ ]:
lr_default = LogisticRegression()
lr_default.fit(X_train, y_train)
print("Prediction on training set:", lr_default.predict(X_train))
print("Accuracy on training set:", lr_default.score(X_train, y_train))
print("default L2 coef, intercept:", lr_default.coef_, lr_default.intercept_)

lr_unreg = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
lr_unreg.fit(X_train, y_train)
print("C=inf coef, intercept:", lr_unreg.coef_, lr_unreg.intercept_)
print("C=inf accuracy:", lr_unreg.score(X_train, y_train))

from matplotlib.lines import Line2D
p_unreg = lr_unreg.predict_proba(grid)[:, 1].reshape(xx.shape)
p_l2 = lr_default.predict_proba(grid)[:, 1].reshape(xx.shape)
fig, ax = plt.subplots(figsize=(5.5, 4.4))
ax.contour(xx, yy, prob, levels=[0.5], colors="#1f77b4", linewidths=2)
ax.contour(xx, yy, p_unreg, levels=[0.5], colors="#2ca02c", linewidths=1.6, linestyles="--")
ax.contour(xx, yy, p_l2, levels=[0.5], colors="#ff7f0e", linewidths=1.6, linestyles=":")
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="#1f77b4", s=90, edgecolor="white", zorder=3)
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="#d62728", marker="x", s=110, zorder=3)
ax.legend(handles=[
    Line2D([0], [0], color="#1f77b4", lw=2, label="from-scratch GD"),
    Line2D([0], [0], color="#2ca02c", lw=1.6, ls="--", label="sklearn C=inf"),
    Line2D([0], [0], color="#ff7f0e", lw=1.6, ls=":", label="sklearn default L2"),
], frameon=False, fontsize=8)
ax.set_xlim(0, 4); ax.set_ylim(0, 3.5)
ax.set_xlabel("$x_0$"); ax.set_ylabel("$x_1$")
ax.set_title("GD vs sklearn boundaries")
plt.show()


## 4. 1-D tumor set


In [ ]:
df1 = pd.read_csv("data/logreg_gd_1d.csv")
x1 = df1["tumor_size"].to_numpy(dtype=float)
y1 = df1["malignant"].to_numpy(dtype=float)
w1, b1, J1 = gradient_descent(x1.reshape(-1, 1), y1, np.zeros(1), 0.0, 0.1, 8000, verbose=False)
print("w, b:", float(w1[0]), b1, "J:", J1[-1])
thresh = -b1 / w1[0]
print("0.5 threshold size:", float(thresh))

fig, ax = plt.subplots(figsize=(6.2, 3.5))
ax.scatter(x1[y1 == 0], y1[y1 == 0], c="#1f77b4", s=80, label="benign")
ax.scatter(x1[y1 == 1], y1[y1 == 1], c="#d62728", marker="x", s=90, label="malignant")
xs = np.linspace(-0.5, 5.5, 200)
ax.plot(xs, sigmoid(w1[0] * xs + b1), color="black", lw=2, label="fitted sigmoid")
ax.axhline(0.5, color="gray", ls="--", lw=0.8)
ax.axvline(thresh, color="#2ca02c", ls=":", lw=1.3, label=f"threshold ≈ {float(thresh):.2f}")
ax.set_xlabel("Tumor size"); ax.set_ylabel("f / label")
ax.set_ylim(-0.08, 1.08); ax.legend(frameon=False, fontsize=8)
ax.set_title("1-D tumor after GD"); ax.grid(True, alpha=0.3)
plt.show()


## 5. More practice


In [ ]:
dfp = pd.read_csv("data/logreg_gd_practice.csv")
Xp = dfp[["x0", "x1"]].to_numpy(dtype=float)
yp = dfp["y"].to_numpy(dtype=float)
wp, bp, Jp = gradient_descent(Xp, yp, np.zeros(2), 0.0, 0.3, 2000, verbose=False)
pred_p = (sigmoid(Xp @ wp + bp) >= 0.5).astype(float)
print("practice GD w,b:", wp, bp, "acc:", np.mean(pred_p == yp), "J:", Jp[-1])

lr_p = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
lr_p.fit(Xp, yp)
print("practice sklearn C=inf acc:", lr_p.score(Xp, yp), "coef:", lr_p.coef_, "b:", lr_p.intercept_)

fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.scatter(Xp[yp == 0, 0], Xp[yp == 0, 1], c="#1f77b4", s=40, label="y=0")
ax.scatter(Xp[yp == 1, 0], Xp[yp == 1, 1], c="#d62728", marker="x", s=50, label="y=1")
xs_line = np.linspace(Xp[:, 0].min() - 0.3, Xp[:, 0].max() + 0.3, 50)
# w0 x + w1 y + b = 0 => y = -(w0 x + b)/w1
ax.plot(xs_line, -(wp[0] * xs_line + bp) / wp[1], color="#1f77b4", lw=1.6, label="GD boundary")
ax.set_title("40-point practice set"); ax.legend(frameon=False, fontsize=8)
ax.grid(True, alpha=0.3); plt.show()


In [ ]:
dfd = pd.read_csv("data/logreg_gd_dti.csv")
xd = dfd["dti"].to_numpy(dtype=float)
yd = dfd["defaulted"].to_numpy(dtype=float)
wd, bd, Jd = gradient_descent(xd.reshape(-1, 1), yd, np.zeros(1), 0.0, 0.8, 8000, verbose=False)
dti_cut = float(-bd / wd[0])
print("DTI model w,b:", float(wd[0]), bd, "J:", Jd[-1])
print("P(default)=0.5 at DTI ≈", dti_cut)
print("train acc:", np.mean(((sigmoid(wd[0] * xd + bd) >= 0.5).astype(float) == yd)))

# analyst sentence
print(
    "Analyst note: on this tiny book, modelled PD crosses 50% near DTI "
    f"{dti_cut:.2f}. Treat as a teaching cutoff, not a policy limit."
)


In [ ]:
# alternate: GD driven only by the vectorized gradient
w_alt, b_alt, _ = gradient_descent(
    X_train, y_train, np.zeros(2), 0.0, 0.1, 10000,
    grad_fn=compute_gradient_logistic_vec, verbose=False,
)
print("alt w,b:", w_alt, b_alt)
print("match Task 2.7?", np.allclose(w_alt, w_out) and np.isclose(b_alt, b_out))


## 6. Simulation


In [ ]:
ALPHAS = [0.01, 0.05, 0.1, 0.5, 1.0]
ITERS_ALPHA = 1500
SIZES = [20, 40, 80, 160]
N_REPS = 20
NOISE = 0.05
ALPHA_MC = 0.3
ITERS_MC = 400
SEED = 7
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.6))
for a in ALPHAS:
    _, _, Jh = gradient_descent(
        X_train, y_train, np.zeros(2), 0.0, a, ITERS_ALPHA, verbose=False
    )
    axes[0].plot(Jh, label=f"α={a}")
axes[0].set_title("J vs iteration by learning rate")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("J")
axes[0].legend(fontsize=8, frameon=False); axes[0].grid(True, alpha=0.3)

acc_mean, acc_std = [], []
for m in SIZES:
    accs = []
    for _ in range(N_REPS):
        half = m // 2
        Xn = np.vstack([
            rng.normal(loc=[1.0, 1.0], scale=0.5, size=(half, 2)),
            rng.normal(loc=[2.7, 2.5], scale=0.5, size=(m - half, 2)),
        ])
        yn = np.array([0] * half + [1] * (m - half), dtype=float)
        flip = rng.random(m) < NOISE
        yn = np.where(flip, 1 - yn, yn)
        wh, bh, _ = gradient_descent(Xn, yn, np.zeros(2), 0.0, ALPHA_MC, ITERS_MC, verbose=False)
        accs.append(np.mean((sigmoid(Xn @ wh + bh) >= 0.5).astype(float) == yn))
    acc_mean.append(np.mean(accs)); acc_std.append(np.std(accs))

axes[1].errorbar(SIZES, acc_mean, yerr=acc_std, marker="o", color="#1f77b4", capsize=3)
axes[1].set_title(f"Monte-Carlo train acc vs m (noise={NOISE})")
axes[1].set_xlabel("m"); axes[1].set_ylabel("mean ± sd acc")
axes[1].set_ylim(0.6, 1.02); axes[1].grid(True, alpha=0.3)
fig.tight_layout(); plt.show()
print("mean acc by m:", list(zip(SIZES, acc_mean)))


## 7. Audience rewrite

**Practitioner.** Batch GD on the six-point lab set with $\alpha=0.1$ and 10 000 steps reaches $w\approx(5.28,5.08)$, $b\approx-14.22$, $J\approx0.017$. The loop gradient matches the published check values to all printed digits, and the vectorized form $X^\top(f-y)/m$ is identical. sklearn with $C=\infty$ also separates the points (acc = 1.0) but stops at a different scale of $w$; default $C=1$ L2 shrinks $|w|$ by roughly an order of magnitude while still fitting these six points. Do not treat coefficient equality as a unit test against `LogisticRegression()`.

**Credit-risk analyst.** Read $f_{w,b}(x)$ as a model PD. On the toy 2-feature book the 50% contour cleanly splits the six files. On the DTI drill the same 50% contour sits near DTI ≈ 0.36 — a teaching cutoff, not a policy limit. Changing the threshold is a risk-appetite lever; changing $C$ in sklearn is a smoothness lever. Report both when you hand a scorecard to underwriting.

**Executive.** We trained a simple yes/no classifier two ways: from-scratch gradient descent and the standard library model. Both classify the six training cases correctly. The library’s default settings quietly shrink the weights (regularization). That is usually desirable on real books and is the reason the two printouts of “the formula” do not match. Next step is a hold-out or time-split test — training accuracy of 100% on six rows is not a production claim.

**Non-specialist.** Imagine two clouds of dots on a page. The algorithm starts with a random dividing line and nudges it, a little at a time, so fewer dots sit on the wrong side. After many tiny nudges the line sits between the two groups. A popular toolkit does the same job in one command; it also prefers a slightly “softer” line so it will not over-react when a new dot arrives.


## 8. Takeaways

- Logistic GD uses the **same** $(f-y)x$ gradient shape as linear regression; only $f$ changed (sigmoid).
- Always clip $z$ and $f$ before `exp` / `log`.
- Cost should fall; if it rises, $\alpha$ is too large.
- sklearn default ≠ unregularized GD. Compare accuracy and the boundary, not raw $w$ on tiny data.
- Threshold, $\alpha$, $C$, and $m$ are the knobs the simulation is meant to make visible.
